# 🎙️ Clone your voice for ALL lessons — Colab

Speaks every lesson's script in **your voice**, all in one run. Only the GPU
step runs here (text → voice). No Whisper, no Demucs, no video upload.

## Just 4 steps:
1. **Runtime → Change runtime type → GPU → Save**
2. Run **Cell 1** (setup — wait ~5 min the first time)
3. Run **Cell 2** → upload your **voice clip** (15–30s, clear)
4. Run **Cell 3** (clones every lesson), then **Cell 4** to download `clones.zip`

Send `clones.zip` to Claude — he builds the final videos.

In [ ]:
# CELL 1 — Setup (GPU check + project + voice engine). ~5 min first time.
!nvidia-smi -L || print('NO GPU: Runtime -> Change runtime type -> GPU, then re-run.')
import os
USER, REPO, BRANCH = 'PrashantRMarathe', 'voice-replacer', 'add-course-video'
if os.path.isdir(REPO):
    !cd {REPO} && git fetch -q && git checkout -q {BRANCH} && git pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/{USER}/{REPO}.git
%cd {REPO}
!pip install -q coqui-tts 'transformers<5' ffmpeg-python soundfile num2words
!pip install -q f5-tts 2>/dev/null || print('(f5-tts optional, skipped)')
print('\n✅ Setup done. Run Cell 2.')

In [ ]:
# CELL 2 — Upload your voice clip (one clean 15-30s file)
from google.colab import files
print('Choose your VOICE clip (mp3/wav)...')
reference_path = list(files.upload().keys())[0]
print('\n✅ Voice:', reference_path, '— run Cell 3.')

In [ ]:
# CELL 3 — Clone your voice for EVERY lesson (model loads once, then loops all)
import os, glob, json, time
os.environ['COQUI_TOS_AGREED'] = '1'
import config, tts
config.TTS_BACKEND = 'f5'   # natural English; use 'xtts' for other languages
config.LANGUAGE = 'en'

lessons = sorted(glob.glob('course_video/src/lessons/*/scripts.json'))
print('Lessons found:', [p.split('/')[-2] for p in lessons])
backend = tts.get_backend()   # load the voice model ONCE for all lessons

t0 = time.time()
for sp in lessons:
    name = sp.split('/')[-2]
    out_dir = f'clones/{name}'
    os.makedirs(out_dir, exist_ok=True)
    scripts = json.load(open(sp, encoding='utf-8'))
    print(f'\n=== {name}: {len(scripts)} scenes ===')
    for i, (key, text) in enumerate(scripts.items(), 1):
        out = f'{out_dir}/{key}.wav'
        if os.path.exists(out):
            print(f'  [{i}/{len(scripts)}] skip {key}'); continue
        backend.synth_to_file(text=text, speaker_wav=reference_path,
                              language=config.LANGUAGE, out_path=out)
        print(f'  [{i}/{len(scripts)}] {key}  ({time.time()-t0:.0f}s)')
print('\n✅ All lessons cloned. Run Cell 4.')

In [ ]:
# CELL 4 — Download everything (one zip, organized by lesson)
!zip -rq clones.zip clones && echo 'zipped clones.zip'
from google.colab import files
files.download('clones.zip')